# 📊 Notebook 1: Exploratory Data Analysis (EDA)

## Fake Instagram Profile Detection System
### Final Year Project — Computer Science, GCUF

---

**Purpose:** Explore and understand the InstaFake dataset before any preprocessing or modeling.

**Dataset:** [fcakyon/instafake-dataset](https://github.com/fcakyon/instafake-dataset) — fake account detection CSV.

**Steps covered:**
1. Import libraries
2. Load dataset
3. Dataset overview
4. Class distribution analysis
5. Feature analysis (per-feature histograms and box plots)
6. Correlation analysis
7. Outlier detection
8. Key findings summary

## 1. Import Libraries

We start by importing all necessary libraries for data analysis and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

# Configuration
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
pd.set_option('display.max_columns', None)

print('✅ All libraries imported successfully!')

## 2. Load the InstaFake Dataset

We load the dataset from the locally cached CSV file. If it doesn't exist, we download it from GitHub.
After loading, we inspect the shape, data types, and first few rows to understand the structure.

In [ ]:
import os
import urllib.request

# Dataset paths
DATA_DIR = os.path.join('..', 'data')
LOCAL_CSV = os.path.join(DATA_DIR, 'instafake_dataset.csv')
DATASET_URL = 'https://raw.githubusercontent.com/fcakyon/instafake-dataset/master/data/fake-account-detection/insta_fake.csv'

# Download if not cached
os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(LOCAL_CSV):
    print('📥 Downloading dataset from GitHub...')
    try:
        urllib.request.urlretrieve(DATASET_URL, LOCAL_CSV)
        print(f'✅ Saved to {LOCAL_CSV}')
    except:
        alt_urls = [
            'https://raw.githubusercontent.com/fcakyon/instafake-dataset/master/data/insta_fake.csv',
            'https://raw.githubusercontent.com/fcakyon/instafake-dataset/main/data/fake-account-detection/insta_fake.csv',
        ]
        for url in alt_urls:
            try:
                urllib.request.urlretrieve(url, LOCAL_CSV)
                print(f'✅ Saved to {LOCAL_CSV}')
                break
            except:
                continue

# Load
df = pd.read_csv(LOCAL_CSV)
print(f'\n📊 Dataset shape: {df.shape}')
print(f'   Rows: {df.shape[0]:,}')
print(f'   Columns: {df.shape[1]}')

In [ ]:
# Display data types
print('Data Types:')
print(df.dtypes)
print(f'\n--- First 10 rows ---')
df.head(10)

## 3. Dataset Overview

Let's get a comprehensive overview of the dataset including:
- Column information and data types
- Descriptive statistics for all numeric columns
- Missing value analysis

In [ ]:
# Dataset info
print('=== Dataset Info ===')
df.info()
print()

In [ ]:
# Descriptive statistics
print('=== Descriptive Statistics ===')
df.describe().T

In [ ]:
# Missing values analysis
missing = df.isnull().sum()
print('=== Missing Values per Column ===')
print(missing)
print(f'\nTotal missing values: {missing.sum()}')

# Missing values heatmap
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df.isnull().T, cbar=True, cmap='YlOrRd', yticklabels=True, ax=ax)
ax.set_title('Missing Values Heatmap', fontsize=14, fontweight='bold')
ax.set_xlabel('Sample Index')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

## 4. Class Distribution

We analyze the distribution of fake vs. genuine profiles to understand if the dataset is balanced or imbalanced.
This is critical because class imbalance affects model training and evaluation.

In [ ]:
# Find label column
label_col = 'is_fake' if 'is_fake' in df.columns else df.columns[-1]
print(f'Label column: {label_col}')
print()

# Class counts
class_counts = df[label_col].value_counts()
print('Class Distribution:')
print(class_counts)
print()

# Percentages
total = len(df)
for cls, count in class_counts.items():
    label = 'Fake' if cls == 1 else 'Genuine'
    print(f'  {label} ({cls}): {count:,} samples ({count/total*100:.1f}%)')

# Check imbalance
minority_pct = class_counts.min() / total * 100
if minority_pct < 40:
    print(f'\n⚠️ Dataset is IMBALANCED — minority class is {minority_pct:.1f}%')
    print('   We will need to use SMOTE or other resampling techniques.')
else:
    print(f'\n✅ Dataset is relatively BALANCED — minority class is {minority_pct:.1f}%')

In [ ]:
# Visualization: Count plot and pie chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
colors = ['#28a745', '#dc3545']
sns.countplot(data=df, x=label_col, ax=axes[0], palette=colors)
axes[0].set_title('Class Distribution — Count Plot', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class (0=Genuine, 1=Fake)')
axes[0].set_ylabel('Count')
for bar in axes[0].patches:
    axes[0].annotate(f'{int(bar.get_height()):,}',
                     (bar.get_x() + bar.get_width()/2., bar.get_height()),
                     ha='center', va='bottom', fontsize=12, fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=['Genuine', 'Fake'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12, 'fontweight': 'bold'},
            explode=(0.05, 0.05), shadow=True)
axes[1].set_title('Class Distribution — Pie Chart', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Feature Analysis

We now analyze each of the key features in the dataset. For each feature, we:
1. Plot a histogram colored by label (fake vs genuine)
2. Plot a box plot split by label
3. Print mean values per class

This helps us understand which features differ significantly between fake and genuine profiles.

In [ ]:
# Identify numeric feature columns (excluding the label)
feature_cols = [col for col in df.select_dtypes(include=[np.number]).columns if col != label_col]
print(f'Numeric features to analyze: {feature_cols}')
print(f'Total: {len(feature_cols)} features')

In [ ]:
# Feature-by-feature analysis
for feature in feature_cols:
    print(f'\n{"="*60}')
    print(f'📊 Feature: {feature}')
    print(f'{"="*60}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram by label
    for cls, color, label in [(0, '#28a745', 'Genuine'), (1, '#dc3545', 'Fake')]:
        subset = df[df[label_col] == cls][feature]
        axes[0].hist(subset, bins=30, alpha=0.6, color=color, label=label, edgecolor='white')

    axes[0].set_title(f'{feature} — Distribution by Label', fontsize=13, fontweight='bold')
    axes[0].set_xlabel(feature)
    axes[0].set_ylabel('Frequency')
    axes[0].legend()

    # Box plot by label
    df_plot = df[[feature, label_col]].copy()
    df_plot[label_col] = df_plot[label_col].map({0: 'Genuine', 1: 'Fake'})
    sns.boxplot(data=df_plot, x=label_col, y=feature, ax=axes[1],
                palette={'Genuine': '#28a745', 'Fake': '#dc3545'})
    axes[1].set_title(f'{feature} — Box Plot by Label', fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Mean values per class
    means = df.groupby(label_col)[feature].mean()
    print(f'  Mean (Genuine): {means.get(0, 0):.4f}')
    print(f'  Mean (Fake):    {means.get(1, 0):.4f}')
    diff = abs(means.get(1, 0) - means.get(0, 0))
    print(f'  Difference:     {diff:.4f}')

## 6. Correlation Analysis

We compute the Pearson correlation matrix to understand linear relationships between features.
This helps identify:
- Features that are highly correlated with the target (useful for prediction)
- Features that are highly correlated with each other (potential multicollinearity)

In [ ]:
# Correlation matrix
corr_matrix = df[feature_cols + [label_col]].corr()

# Heatmap
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, ax=ax, mask=mask, linewidths=0.5,
    square=True, cbar_kws={'shrink': 0.8},
    vmin=-1, vmax=1
)
ax.set_title('Pearson Correlation Heatmap', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Top 3 features most correlated with the label
label_corr = corr_matrix[label_col].drop(label_col).abs().sort_values(ascending=False)

print('🎯 Top 3 Features Most Correlated with Label:')
for i, (feat, val) in enumerate(label_corr.head(3).items()):
    direction = 'positive' if corr_matrix[label_col][feat] > 0 else 'negative'
    print(f'  {i+1}. {feat}: |r| = {val:.4f} ({direction} correlation)')

print('\n📝 Note: Higher absolute correlation with the label suggests stronger predictive power.')

# Visualize
fig = go.Figure(go.Bar(
    x=label_corr.values,
    y=label_corr.index,
    orientation='h',
    marker_color='#1a73e8'
))
fig.update_layout(
    title='Feature Correlation with Label (Absolute Values)',
    xaxis_title='|Correlation|',
    yaxis_title='Feature',
    template='plotly_white',
    height=400
)
fig.show()

## 7. Outlier Detection

We use box plots to identify outliers in key numeric features.
Outliers can significantly affect model performance and may need to be handled during preprocessing.

In [ ]:
# Outlier detection for key features
outlier_features = [col for col in feature_cols if 'count' in col.lower() or 'length' in col.lower()]
if not outlier_features:
    outlier_features = feature_cols[:4]  # fallback to first 4 features

n_features = len(outlier_features)
fig, axes = plt.subplots(1, n_features, figsize=(5 * n_features, 6))
if n_features == 1:
    axes = [axes]

for i, feature in enumerate(outlier_features):
    sns.boxplot(data=df, y=feature, ax=axes[i], color='#1a73e8', width=0.4)
    axes[i].set_title(f'{feature}', fontsize=12, fontweight='bold')
    
    # Count outliers using IQR method
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[feature] < Q1 - 1.5*IQR) | (df[feature] > Q3 + 1.5*IQR)]
    axes[i].set_xlabel(f'Outliers: {len(outliers)}')

plt.suptitle('Outlier Detection — Box Plots', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print outlier summary
print('\n=== Outlier Summary (IQR Method) ===')
for feature in outlier_features:
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[feature] < Q1 - 1.5*IQR) | (df[feature] > Q3 + 1.5*IQR)]
    print(f'  {feature}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)')

## 8. Key Findings Summary

After conducting the exploratory data analysis, here are our key observations:

1. **Dataset Composition:** The dataset contains labeled Instagram profiles with features capturing follower counts, following counts, post counts, profile picture presence, account privacy, bio length, username length, and digit count in username.

2. **Class Distribution:** The dataset shows the distribution of fake vs genuine profiles. If imbalanced, we will apply SMOTE during preprocessing.

3. **Follower/Following Patterns:** Fake profiles tend to have different follower-to-following ratios compared to genuine profiles. Genuine profiles typically have more followers relative to following.

4. **Profile Completeness:** Fake profiles are more likely to lack profile pictures and have shorter biographies, indicating incomplete profiles.

5. **Username Patterns:** Fake profiles often have more digits in their usernames, suggesting auto-generated account names.

6. **Correlation Insights:** The features most strongly correlated with the fake label provide the best signals for our classifier.

7. **Outliers:** Some features have significant outliers that we need to consider during preprocessing — especially follower and following counts which can span several orders of magnitude.

---

**Next Steps:**
- Proceed to **Notebook 02: Preprocessing** to clean the data, engineer features, and prepare it for model training.
- Address class imbalance with SMOTE.
- Normalize features using StandardScaler.